# Trajectories of Change Quickstart

This notebook runs the quick start from the README on the bundled example dataset, then looks into one author in depth, down to the individual words that carry the divergence. The final sections show how to switch to your own data.

## Install

In Colab, this installs the package with plotting extras. In a local environment where the package is already importable, the cell only prints the version. The pin matches the release this notebook ships with. Remove the pin to get the latest version.

In [ ]:
try:
    import trajectories_of_change
except ImportError:
    %pip install "trajectories-of-change[plotting]==0.2.0"
    import trajectories_of_change

print(f"trajectories_of_change {trajectories_of_change.__version__}")

## Load the example dataset

The bundled synthetic dataset has prepared `publications.parquet` and `references.parquet`. Outside a repository checkout, the cell downloads the same files from the v0.2.0 release tag. Every author in it behaves in a known way by construction, so each result below is verifiable.

In [ ]:
from pathlib import Path
import urllib.request

from trajectories_of_change import load_dataset_bundle, run_metric, run_metrics

data_dir = Path("examples/data")
if not data_dir.exists():
    data_dir = Path("data/example")
    data_dir.mkdir(parents=True, exist_ok=True)
    base_url = "https://raw.githubusercontent.com/raphschlatt/Trajectories_of_Change/v0.2.0/examples/data"
    for name in ["publications.parquet", "references.parquet"]:
        target = data_dir / name
        if not target.exists():
            urllib.request.urlretrieve(f"{base_url}/{name}", target)

bundle = load_dataset_bundle(
    data_dir / "publications.parquet",
    data_dir / "references.parquet",
)
print(f"publications: {len(bundle.publications)}")
print(f"references: {len(bundle.references)}")

## The five most productive authors

`run_metrics` computes all four measures for a set of authors and returns one row per author. This is the quick start from the README, executed live. The `*_level` columns are each author's average divergence, or density, across time slices.

In [ ]:
metrics = run_metrics(bundle, top_n=5, show_progress=False)

levels = ["vocab_kld_all_level", "ref_vocab_kld_all_level", "cocit_kld_all_level", "density_neglog_level"]
metrics[["author_display_name"] + levels].round(2)

You should see the pattern of the synthetic dataset: every author is unremarkable except in the one column of the behavior built into it.

- "Stable Vocabulary, V." holds a stably divergent vocabulary, `vocab_kld_all_level` around 13.8 bits against a 0.03 baseline. `kld_all` sums all feature contributions, so values above 1 bit are normal.
- "Citation Distinct, C." co-cites differently (`cocit` around 18.1) and consequently also cites differently worded literature (`ref_vocab` around 2.5).
- "Density Shift, D." moves into a sparser field region (`density` around 2.7, and lower values mean denser neighbourhoods).
- "Spiky Vocabulary, S." looks unremarkable in the levels on purpose. Its divergence comes as single-slice spikes rather than a raised average.
- "Field-Like, F." is the baseline.

## One author in depth: the words behind the divergence

`run_metric` computes a single measure for a single author and returns the full per-slice detail. Two plots tell the story. The first shows the divergence trajectory itself. The second follows the individual terms behind it over time, because for Own Vocabulary the `pointwise` table attributes every slice's divergence to concrete words.

In [ ]:
import plotly.express as px

result = run_metric(bundle, metric="own_vocab", target_author_uid="uid:stable_vocab_distinct", include_async=False)

fig = px.line(result.sync, x="slice", y="kld_all", markers=True,
              title="Own Vocabulary divergence: Stable Vocabulary, V.")
fig.update_yaxes(range=[0, float(result.sync["kld_all"].max()) * 1.15], title="KLD_all (bits)")
fig.update_xaxes(title="Timeslice (End Year)")
fig.show()

totals = result.pointwise.groupby("term")["kld_contribution"].sum()
top_terms = totals.abs().sort_values(ascending=False).head(6).index
per_slice = result.pointwise[result.pointwise["term"].isin(top_terms)]

fig = px.line(per_slice, x="slice", y="kld_contribution", color="term", markers=True,
              title="Term contributions over time")
fig.update_yaxes(title="KLD contribution (bits)")
fig.update_xaxes(title="Timeslice (End Year)")
fig.show()

print("Summed contribution per term:")
print(totals.loc[top_terms].round(2).to_string())

The trajectory is flat at about 13.8 bits in every time slice, which is exactly what "stable divergence" means. The axis starts at zero, so what you see is the real magnitude of the divergence, not zoomed noise. The term lines show why: the four marker terms `frame`, `tetrad`, `torsion` and `gauge` contribute about 3.5 bits in every single slice, while common field terms such as `field` or `equation` sit slightly below zero throughout, because the author uses them less than the field does. This is where a divergence stops being a number and becomes a vocabulary you can read, together with how it develops over time.

## More to try

The dataset contains three more synthetic authors (`uid:correlated_distinct`, `uid:converging_distinct`, `uid:geometry_trap`), and every measure works the same way. Uncomment a line:

In [ ]:
# run_metric(bundle, metric="citation_identity", target_author_uid="uid:citation_distinct").sync
# run_metric(bundle, metric="density", target_author_uid="uid:geometry_trap").sync
# run_metrics(bundle, top_n=8)  # adds the remaining synthetic authors
#
# The package also ships full reporting dashboards, built for real corpora:
# per-author sync, pointwise, and async lead/lag figures, and cohort overview
# figures across many authors.
# from trajectories_of_change.plotting import plot_metric, plot_multimetric
# result_full = run_metric(bundle, metric="own_vocab", target_author_uid="uid:stable_vocab_distinct")
# _ = plot_metric(result_full, show=True)
# _ = plot_multimetric(metrics, show=True)

## CLI Parity

The CLI exposes the same simple paths:

```bash
toc metric own_vocab publications.parquet references.parquet --target-author-uid AUTHOR_UID --out-dir outputs/metric
toc plot metric outputs/metric --out-dir outputs/metric/figures
toc metrics publications.parquet references.parquet --target AUTHOR_UID --out outputs/metrics.parquet
toc plot multimetric outputs/metrics.parquet --out-dir outputs/plots
```

## Use Your Own Data

Swap the example files for your own prepared bundle:

1. Bring two Parquet files that satisfy the data contract: `publications.parquet` (`Bibcode`, `Year`, `Author`, `References`; plus `tokens`, `embedding_2d_x`/`embedding_2d_y`, and `author_uids` for the full measure set) and `references.parquet` (`Bibcode`, `Author`).
2. For raw exports (for example from ADS, with author-name disambiguation applied), run `toc prepare raw_publications.parquet raw_references.parquet --out-dir prepared/` once; it normalizes IDs, deduplicates, and cleans the author-identity layer.
3. Replace the paths in the load cell above with your prepared files and re-run the notebook.
4. Find target UIDs with `bundle.publications["author_uids"].explode().value_counts().head(20)`.

Full input format: [docs/data_contract.md](https://github.com/raphschlatt/Trajectories_of_Change/blob/main/docs/data_contract.md)